# Optiver Realized Volatility — Exploration

In [1]:
import pandas as pd

train = pd.read_csv("data/train.csv")
print(train.shape)
print(train.columns)
train.head()

(428932, 3)
Index(['stock_id', 'time_id', 'target'], dtype='str')


,stock_id,time_id,target
0,0,5,0.004136
1,0,11,0.001445
2,0,16,0.002168
3,0,31,0.002195
4,0,62,0.001747


In [2]:
# book_train.parquet is partitioned by stock_id; only loading stock 0 for now.
book_stock_0 = pd.read_parquet("data/book_train.parquet/stock_id=0")
print(book_stock_0.shape)
print(book_stock_0.columns)
book_stock_0.head()

(917553, 10)
Index(['time_id', 'seconds_in_bucket', 'bid_price1', 'ask_price1',
       'bid_price2', 'ask_price2', 'bid_size1', 'ask_size1', 'bid_size2',
       'ask_size2'],
      dtype='str')


,time_id,seconds_in_bucket,bid_price1,ask_price1,bid_price2,ask_price2,bid_size1,ask_size1,bid_size2,ask_size2
0,5,0,1.001422,1.002301,1.00137,1.002353,3,226,2,100
1,5,1,1.001422,1.002301,1.00137,1.002353,3,100,2,100
2,5,5,1.001422,1.002301,1.00137,1.002405,3,100,2,100
3,5,6,1.001422,1.002301,1.00137,1.002405,3,126,2,100
4,5,7,1.001422,1.002301,1.00137,1.002405,3,126,2,100


In [3]:
one_window=book_stock_0[book_stock_0["time_id"]==5]
print(one_window.head())

   time_id  seconds_in_bucket  bid_price1  ask_price1  bid_price2  ask_price2  \
0        5                  0    1.001422    1.002301     1.00137    1.002353   
1        5                  1    1.001422    1.002301     1.00137    1.002353   
2        5                  5    1.001422    1.002301     1.00137    1.002405   
3        5                  6    1.001422    1.002301     1.00137    1.002405   
4        5                  7    1.001422    1.002301     1.00137    1.002405   

   bid_size1  ask_size1  bid_size2  ask_size2  
0          3        226          2        100  
1          3        100          2        100  
2          3        100          2        100  
3          3        126          2        100  
4          3        126          2        100  


In [4]:
# numerator: cross-weight each side's price by the OTHER side's size, then add
one_window["wap"]=(
     (one_window["bid_price1"] * one_window["ask_size1"]  # best buy price weighted by sell-side size
     + one_window["ask_price1"] * one_window["bid_size1"])  # + best sell price weighted by buy-side size
    / (one_window["bid_size1"] + one_window["ask_size1"])  # divide by total size at the best bid+ask
)
import numpy as np
#return is how much the price moved from one moment to the next 
#we use log because it is additive and symmetric so a move up then an equal move down will cancel to 0
one_window["log_return"]=np.log(one_window["wap"]/one_window["wap"].shift(1))
print(one_window[["seconds_in_bucket", "wap", "log_return"]].head())

   seconds_in_bucket       wap  log_return
0                  0  1.001434         NaN
1                  1  1.001448    0.000014
2                  5  1.001448    0.000000
3                  6  1.001443   -0.000005
4                  7  1.001443    0.000000


In [5]:
#realized volatility
realized_vol=np.sqrt((one_window["log_return"]**2).sum())
print("realized vol (computed):", realized_vol)
#get row with both stock_id=0 and time_id=5 for target
target_row=train[(train["stock_id"]==0) & (train["time_id"]==5)]
target_value=target_row["target"].iloc[0]
print("target from train.csv: ", target_value )

realized vol (computed): 0.004499364172786568
target from train.csv:  0.004135767


In [6]:
#WAP for every row, not just one window
book_stock_0["wap"]=(book_stock_0["bid_price1"]*book_stock_0["ask_size1"]+
                     book_stock_0["ask_price1"] * book_stock_0["bid_size1"]) / (book_stock_0["bid_size1"]+ book_stock_0["ask_size1"])
book_stock_0[["bid_price1", "ask_price1", "wap"]].head()

#log return per row, shifted within each time_id window only
#groupby prevents comparing the last row of one window to the first row of the next 
book_stock_0["log_return"]=np.log(book_stock_0["wap"]/ book_stock_0.groupby("time_id")["wap"].shift(1))

#realized vol=sqrft(sum of squared log returns), one value per time_id window 
realized_vol_per_window=(book_stock_0.groupby("time_id")["log_return"]
                         .apply(lambda x: np.sqrt((x**2).sum()))
                         .rename("realized_vol"))
print(realized_vol_per_window.head())

#sanity check 
stock_0_targets=train[train["stock_id"]==0].set_index("time_id")["target"]
comparison=pd.concat([realized_vol_per_window, stock_0_targets], axis=1)
print(comparison.head(10)) # close in magnitude and move together 


time_id
5     0.004499
11    0.001204
16    0.002369
31    0.002574
62    0.001894
Name: realized_vol, dtype: float64
         realized_vol    target
time_id                        
5            0.004499  0.004136
11           0.001204  0.001445
16           0.002369  0.002168
31           0.002574  0.002195
62           0.001894  0.001747
72           0.007902  0.004912
97           0.010034  0.009388
103          0.005331  0.004120
109          0.001797  0.002182
123          0.003273  0.002669


Volatility clustering = naive baseline 

In [7]:
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(((y_true-y_pred)/y_true)**2))

score_stock_0=rmspe(comparison["target"], comparison["realized_vol"])
print("RSMPE for stock 0 naive baseline: ", score_stock_0)

RSMPE for stock 0 naive baseline:  0.39322701284497674


In [8]:
def compute_realized_vol_for_stock(stock_id):
    book = pd.read_parquet(f"data/book_train.parquet/stock_id={stock_id}")
    book["wap"] = (
        (book["bid_price1"] * book["ask_size1"] + book["ask_price1"] * book["bid_size1"])
        / (book["bid_size1"] + book["ask_size1"])
    )
    book["log_return"] = np.log(book["wap"] / book.groupby("time_id")["wap"].shift(1))
    realized_vol = (
        book.groupby("time_id")["log_return"]
        .apply(lambda x: np.sqrt((x**2).sum()))
        .rename("realized_vol")
        .reset_index()
    )
    realized_vol["stock_id"] = stock_id
    return realized_vol

In [9]:
subset_stock_ids = [0, 1, 2, 3]
all_predictions = pd.concat(
    [compute_realized_vol_for_stock(s) for s in subset_stock_ids],
    ignore_index=True,
)
print(all_predictions.shape)
all_predictions.head()

(15320, 3)


,time_id,realized_vol,stock_id
0,5,0.004499,0
1,11,0.001204,0
2,16,0.002369,0
3,31,0.002574,0
4,62,0.001894,0


In [10]:
merged = all_predictions.merge(train, on=["stock_id", "time_id"])
naive_baseline_rmspe = rmspe(merged["target"], merged["realized_vol"])
print("Naive baseline RMSPE across stocks", subset_stock_ids, ":", naive_baseline_rmspe)

Naive baseline RMSPE across stocks [0, 1, 2, 3] : 0.33583164444718683


The naive baseline predictions are off by aprox. 34% on average. I will consider this my new benchmark and try to get better results with a LightGBM model 